In [33]:
!pip install torch torchvision torchaudio
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import math
from collections import Counter

## Real dataset

In [34]:
FEATURE_COLUMNS = ['beta_propensity', 'proline_fraction', 'net_charge', 'polar_fraction'] # 'AAT', 'TA', 'a3vSA' <- nie ma ich tu bo nie można obliczyć na bieżąco w treningu

path = "files/df_ml_good_with_features.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(1934, 21)


,id,sequence,length,class,merge_id,a3vSA,AAT,Na4vSS,THSA,nHS,...,net_charge,hydrophobicity,polar_fraction,nonpolar_fraction,acidic_fraction,basic_fraction,aromatic_fraction,beta_propensity,seq_entropy,proline_fraction
0,GI_171848907_PDB_2RNM_A__1_1,GRNSAKDIRTEERARVQLGNV,21,1,GI_171848907_PDB_2RNM_A__1_1_0,-0.457,0.190,-0.494,0.190,1.0,...,2.0,-1.185714,0.238095,0.380952,0.142857,0.238095,0.000000,0.953333,3.535175,0.000000
1,GI_342871650_GB_EGU74155_1_1,VRIYAKDIKSEEMARVRVGNE,21,1,GI_342871650_GB_EGU74155_1_1_0,-0.160,2.140,-0.165,1.871,1.0,...,1.0,-0.676190,0.142857,0.428571,0.190476,0.238095,0.047619,0.990000,3.427333,0.000000
2,GI_342887385_GB_EGU86897_1_1,GKNSAGRINGPGMVNIGNS,19,1,GI_342887385_GB_EGU86897_1_1_0,-0.256,1.438,-0.256,1.438,1.0,...,2.0,-0.563158,0.315789,0.578947,0.000000,0.105263,0.000000,0.937368,3.005315,0.052632
3,GI_347837243_EMB_CCD5181_1_1,HRIKIGKVTQASNAKAVIGVH,21,1,GI_347837243_EMB_CCD5181_1_1_0,-0.001,4.054,-0.008,4.054,2.0,...,4.2,-0.019048,0.190476,0.523810,0.000000,0.285714,0.000000,1.081429,3.296148,0.000000
4,GI_475677570_GB_EMT74561_1_1,VRNYASEIKGDEDAKVRLGND,21,1,GI_475677570_GB_EMT74561_1_1_0,-0.436,0.570,-0.435,0.000,0.0,...,-1.0,-1.138095,0.190476,0.380952,0.238095,0.190476,0.047619,0.912381,3.499228,0.000000


Dataset

In [35]:
MAX_LEN = 50

class ProteinDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        seq = encode_sequence(row["sequence"], MAX_LEN)

        features = torch.tensor(
            row[FEATURE_COLUMNS].values.astype(float),
            dtype=torch.float32
        )

        return seq, features

In [36]:
scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS])

Tokenizacja

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 = padding
idx_to_aa = {i+1: aa for i, aa in enumerate(AMINO_ACIDS)}

PAD_IDX = 0
VOCAB_SIZE = len(AMINO_ACIDS) + 1


def encode_sequence(seq, max_len=50):
    seq = seq[:max_len]
    encoded = [aa_to_idx.get(a, 0) for a in seq]
    encoded += [PAD_IDX] * (max_len - len(encoded))
    return torch.tensor(encoded)


def decode_sequence(indices):
    return "".join([idx_to_aa.get(i, "") for i in indices if i != 0])

Positional encoding

In [38]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

cVAE

In [39]:
class CVAE_Transformer(nn.Module):
    def __init__(self,
                 vocab_size,
                 seq_len=50,
                 d_model=128,
                 nhead=4,
                 num_layers=2,
                 latent_dim=64,
                 feature_dim=10):
        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model
        self.latent_dim = latent_dim

        # === EMBEDDING ===
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, seq_len)

        # === ENCODER (Transformer) ===
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        # === FEATURES ===
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # === LATENT ===
        self.fc_mu = nn.Linear(d_model + 32, latent_dim)
        self.fc_logvar = nn.Linear(d_model + 32, latent_dim)

        # === DECODER INPUT ===
        self.decoder_input = nn.Linear(latent_dim + feature_dim, d_model)

        # === DECODER (Transformer) ===
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)

        self.output_layer = nn.Linear(d_model, vocab_size)

    def encode(self, x, features):
        x = self.embedding(x)
        x = self.pos_enc(x)

        x = x.permute(1, 0, 2)  # (seq, batch, dim)
        h = self.transformer_encoder(x)
        h = h.mean(dim=0)  # global pooling

        f = self.feature_mlp(features)

        h = torch.cat([h, f], dim=1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, features, tgt):
        z = torch.cat([z, features], dim=1)
        z = self.decoder_input(z)

        memory = z.unsqueeze(0).repeat(self.seq_len, 1, 1)

        tgt = self.embedding(tgt)
        tgt = self.pos_enc(tgt)
        tgt = tgt.permute(1, 0, 2)

        out = self.transformer_decoder(tgt, memory)
        out = out.permute(1, 0, 2)

        return self.output_layer(out)

    def forward(self, x, features):
        mu, logvar = self.encode(x, features)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z, features, x)
        return logits, mu, logvar

Inicjalizacja modelu

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CVAE_Transformer(
    vocab_size=VOCAB_SIZE,
    seq_len=MAX_LEN,
    feature_dim=len(FEATURE_COLUMNS)
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

C:\Users\marts\AppData\Local\Temp\ipykernel_37272\615530187.py:22: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)


Loss

In [41]:
def loss_function(logits, targets, mu, logvar, beta=0.1):
    recon_loss = F.cross_entropy(
        logits.view(-1, VOCAB_SIZE),
        targets.view(-1),
        ignore_index=PAD_IDX
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + beta * kl, recon_loss, kl

Cechy do treningu

In [42]:
def compute_features(seq):
    L = len(seq)
    if L == 0:
        return torch.zeros(4)

    # amino groups
    aa_groups = {
        "polar": "NQSTY",
    }

    def aa_fraction(seq, aas):
        return sum(seq.count(a) for a in aas) / L

    # beta propensity
    beta_prop = {
        'A':0.83,'C':1.19,'D':0.54,'E':0.37,'F':1.38,'G':0.75,'H':0.87,'I':1.60,
        'K':0.74,'L':1.30,'M':1.05,'N':0.89,'P':0.55,'Q':1.10,'R':0.93,'S':0.75,
        'T':1.19,'V':1.70,'W':1.37,'Y':1.47
    }

    beta_propensity = sum(beta_prop.get(a, 0) for a in seq) / L

    # proline fraction
    proline_fraction = seq.count("P") / L

    # net charge
    pos = sum(a in "KRH" for a in seq)
    neg = sum(a in "DE" for a in seq)
    net_charge = pos - neg

    # polar fraction
    polar_fraction = aa_fraction(seq, aa_groups["polar"])

    return torch.tensor([
        beta_propensity,
        proline_fraction,
        net_charge,
        polar_fraction
    ], dtype=torch.float32)


Generowanie

In [43]:
def generate(model, features, max_len=50, device="cpu"):
    model.eval()

    with torch.no_grad():
        z = torch.randn(1, model.latent_dim).to(device)
        features = features.unsqueeze(0).to(device)

        seq = torch.zeros((1, max_len), dtype=torch.long).to(device)

        for i in range(max_len):
            logits = model.decode(z, features, seq)
            probs = F.softmax(logits[:, i], dim=-1)
            aa = torch.multinomial(probs, 1)
            seq[:, i] = aa.squeeze()

        return decode_sequence(seq[0].cpu().numpy())

Trening

In [44]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = ProteinDataset(train_df)
val_dataset = ProteinDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_loss = float("inf")
patience = 7
counter = 0

EPOCHS = 50
alpha = 1.0
lambda_len = 0.3

for epoch in range(EPOCHS):

    beta = min(0.5, epoch / 10)   # KL (more stable)

    # TRAIN
    model.train()
    train_loss = 0

    for x, f in train_loader:
        x, f = x.to(device), f.to(device)

        # SHIFT (key!)
        input_seq = x[:, :-1]
        target_seq = x[:, 1:]

        opt.zero_grad()

        logits, mu, logvar = model(input_seq, f)

        #RECON LOSS
        recon = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_seq.reshape(-1),
            ignore_index=0
        )

        # KL
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

        # FEATURE LOSS
        pred_idx = logits.argmax(-1).detach()

        f_loss = 0
        valid_count = 0

        for i in range(pred_idx.size(0)):
            seq = decode_sequence(pred_idx[i].cpu().numpy())

            if len(seq) == 0:
                continue

            pf = compute_features(seq)

            # FIX scaler
            pf_df = pd.DataFrame([pf.numpy()], columns=FEATURE_COLUMNS)
            pf_scaled = scaler.transform(pf_df)
            pf = torch.tensor(pf_scaled[0], dtype=torch.float32)

            f_loss += F.mse_loss(pf, f[i])
            valid_count += 1

        if valid_count > 0:
            f_loss /= valid_count
        else:
            f_loss = torch.tensor(0.0).to(device)

        # LENGTH LOSS
        pred_len = (pred_idx != 0).sum(dim=1).float()
        true_len = (target_seq != 0).sum(dim=1).float()

        length_loss = (pred_len - true_len).abs().mean()

        #TOTAL LOSS
        loss = recon + beta * kl + alpha * f_loss + lambda_len * length_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, f in val_loader:
            x, f = x.to(device), f.to(device)

            input_seq = x[:, :-1]
            target_seq = x[:, 1:]

            logits, mu, logvar = model(input_seq, f)

            recon = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                target_seq.reshape(-1),
                ignore_index=0
            )

            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

            loss = recon + beta * kl
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1} | Train {train_loss:.4f} | Val {val_loss:.4f}")

    #EARLY STOP
    if val_loss < best_loss - 0.001:
        best_loss = val_loss
        counter = 0
        best_state = model.state_dict()
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break


model.load_state_dict(best_state)

save_path = "files/models/cvae_protein_model.pt"
torch.save(model.state_dict(), save_path)

Epoch 1 | Train 22.5206 | Val 2.3838
Epoch 2 | Train 26.2320 | Val 1.6585
Epoch 3 | Train 21.7515 | Val 0.3157
Epoch 4 | Train 28.5308 | Val 0.1216
Epoch 5 | Train 40.7116 | Val 0.0657
Epoch 6 | Train 33.1303 | Val 0.0482
Epoch 7 | Train 31.6686 | Val 0.0279
Epoch 8 | Train 27.0999 | Val 0.0182
Epoch 9 | Train 22.2109 | Val 0.0133
Epoch 10 | Train 19.7557 | Val 0.0114
Epoch 11 | Train 15.2381 | Val 0.0128
Epoch 12 | Train 15.2331 | Val 0.0084
Epoch 13 | Train 18.4164 | Val 0.0079
Epoch 14 | Train 19.0837 | Val 0.0053
Epoch 15 | Train 17.5355 | Val 0.0041
Epoch 16 | Train 20.8251 | Val 0.0025
Epoch 17 | Train 17.7346 | Val 0.0074
Epoch 18 | Train 15.2909 | Val 0.0037
Epoch 19 | Train 26.9035 | Val 0.0046
Epoch 20 | Train 21.2929 | Val 0.0052
Epoch 21 | Train 15.0472 | Val 0.0053
Epoch 22 | Train 18.5885 | Val 0.0031
Epoch 23 | Train 16.6091 | Val 0.0045
Early stopping at epoch 23


## Generating

In [47]:
best_state = "files/models/cvae_protein_model.pt"
state_dict = torch.load(best_state, map_location="cpu")
model.load_state_dict(state_dict)

<All keys matched successfully>

In [48]:
sample_features = torch.tensor(df[FEATURE_COLUMNS].iloc[0].values, dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated:", generated_seq)

Generated: VSSDKRWLGDDVHHGMDDDHHWLWISVHHHSDGHHHHLHKHAHHHHHHHH


In [31]:
# Oryginalna sekwencja z pierwszego wiersza
original_seq = df["sequence"].iloc[0]
print("Original sequence:", original_seq)

# Wygenerowana sekwencja
sample_features = torch.tensor(df[FEATURE_COLUMNS].iloc[0].values, dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated sequence:", generated_seq)

Original sequence: GRNSAKDIRTEERARVQLGNV
Generated sequence: 


In [23]:
# oryginalna sekwencja
original_seq = df["sequence"].iloc[0]
print("Original sequence:", original_seq)

# wygenerowana sekwencja
sample_features = torch.tensor(df[FEATURE_COLUMNS].iloc[0].values, dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated sequence:", generated_seq)

# funkcja do cech (już była wcześniej)
def compute_features_from_sequence(seq):
    aa_list = list(seq)
    L = len(aa_list)

    kd = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,
          'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,
          'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}

    hydrophobicity = sum([kd.get(a,0) for a in aa_list])/L
    pos = sum(a in "KRH" for a in aa_list)
    neg = sum(a in "DE" for a in aa_list)
    charge = pos - neg
    from collections import Counter
    counts = Counter(aa_list)
    probs = torch.tensor([counts.get(a,0)/L for a in AMINO_ACIDS], dtype=torch.float32)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8))
    aromatic_fraction = sum(a in "FWY" for a in aa_list)/L
    proline_fraction = sum(a=="P" for a in aa_list)/L
    pI = (pos + 1) / (neg + 1)
    instability_index = sum(a in "DEKPQR" for a in aa_list)/L

    return {
        "hydrophobicity": hydrophobicity,
        "charge": charge,
        "entropy": entropy.item(),
        "aromatic_fraction": aromatic_fraction,
        "proline_fraction": proline_fraction,
        "pI": pI,
        "instability_index": instability_index
    }

# 🔹 Obliczenie cech
orig_features = compute_features_from_sequence(original_seq)
gen_features = compute_features_from_sequence(generated_seq)

# 🔹 Porównanie w tabeli
import pandas as pd
comparison_df = pd.DataFrame([orig_features, gen_features], index=["Original", "Generated"])
print("\nFeature comparison:")
print(comparison_df)

Original sequence: GRNSAKDIRTEERARVQLGNV
Generated sequence: 


ZeroDivisionError: division by zero